# 实验 03：逻辑回归与贷款违约预测
主讲：钟志诚；整理：曾有成；中国科学技术大学 人工智能与数据科学学院。

按 C0–C7 执行。主要补全 `student.py`，每次修改后**重启内核并从头运行到当前阶段**。所有 `None` 是占位，不是正确实现。随包为天池真实贷款数据的固定课堂子集，重新划分为 8000/1000/1000，并非教材原有抽样。普通脚本与本 Notebook 共用一份学生代码。

先阅读前置讲义与实验手册，在 `code/` 目录打开本文件。

In [ ]:
import sys
from pathlib import Path
import torch
import pandas as pd
import student
import support
import checkpoints
import workflow
print(sys.executable)
print(torch.__version__, pd.__version__)
torch.set_num_threads(1)

## C0：观察形状
预测矩阵乘法输出，再将权重 1 改为 2。此段不需要补全。

**报告素材：**记录运行入口与版本、输入和权重的形状，以及修改前后的预期和实际得分，解释变化原因。

In [ ]:
X_demo = torch.tensor([[1., 2.], [1., -1.]])
w_demo = torch.tensor([[0.], [1.]])
print(X_demo @ w_demo)
checkpoints.check(student, 0)

## C1：读表与编码
补 filter_labels、encode_employment；解释删除标签缺失行与填充输入的区别。

In [ ]:
raw = support.load_split(support.DATA_DIR, "train")
print(raw.head())
print(raw.isna().sum())
checkpoints.check(student, 1)

**报告素材：**记录对应函数与关键代码、小例子的输入及预期/实际输出，并解释结果。各阶段具体要求见 `report-template.md` 第 2 节；C6 另保留一次参数更新的手算对照。

## C2：训练统计量与张量
补 fit_statistics、apply_statistics、to_tensors；验证值 100 不能参与均值估计。

In [ ]:
checkpoints.check(student, 2)
train, val, state = support.prepare_train_val(student, support.DATA_DIR)
print(train[0].shape, train[1].shape)
print(val[0].shape, val[1].shape)

**报告素材：**记录对应函数与关键代码、小例子的输入及预期/实际输出，并解释结果。各阶段具体要求见 `report-template.md` 第 2 节；C6 另保留一次参数更新的手算对照。

## C3：成对取样与批次
补 Dataset 的两个方法与 make_loader；最后一批不丢弃。

In [ ]:
checkpoints.check(student, 3)
loader = student.make_loader(student.LoanDataset(*train), 64, True)
print([len(y) for X, y in loader])

**报告素材：**记录对应函数与关键代码、小例子的输入及预期/实际输出，并解释结果。各阶段具体要求见 `report-template.md` 第 2 节；C6 另保留一次参数更新的手算对照。

## C4：得分与概率
补 sigmoid、参数初始化与 forward；说明 @ 与 * 的区别。

In [ ]:
checkpoints.check(student, 4)
model = student.LogisticRegression(train[0].shape[1])
z = model(train[0][:3])
print(z, student.sigmoid(z))

**报告素材：**记录对应函数与关键代码、小例子的输入及预期/实际输出，并解释结果。各阶段具体要求见 `report-template.md` 第 2 节；C6 另保留一次参数更新的手算对照。

## C5：损失与正则化
补 data_loss 和 penalty；得分全零时平均交叉熵应约为 0.693147。L2 惩罚使用权重平方和的一半。

In [ ]:
checkpoints.check(student, 5)
print(student.data_loss(model(train[0][:3]), train[1][:3]))

**报告素材：**记录对应函数与关键代码、小例子的输入及预期/实际输出，并解释结果。各阶段具体要求见 `report-template.md` 第 2 节；C6 另保留一次参数更新的手算对照。

## C6：训练更新
补 train_epoch；先手算两条样本，再检查连续两次更新。

In [ ]:
checkpoints.check(student, 6)

**报告素材：**记录对应函数与关键代码、小例子的输入及预期/实际输出，并解释结果。各阶段具体要求见 `report-template.md` 第 2 节；C6 另保留一次参数更新的手算对照。

## C7：验证选择
补 binary_metrics；再运行三组预设候选。此段不读取 test.csv。

In [ ]:
checkpoints.check(student, 7)
output = Path("outputs-tianchi-balanced-notebook")
output.mkdir(exist_ok=True)
workflow.train_candidates(student, support.DATA_DIR, output)
print(pd.read_csv(output / "candidates.csv"))
print(pd.read_csv(output / "validation_thresholds.csv"))

**报告素材：**记录对应函数与关键代码、小例子的输入及预期/实际输出，并解释结果。各阶段具体要求见 `report-template.md` 第 2 节；C6 另保留一次参数更新的手算对照。

## 冻结与最终测试
先记录正则化方案、系数、轮数与选择依据。主实验阈值固定为 0.5。然后把开关改为 True，执行一次最终测试；测试后不再据此调参。

In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    workflow.final_test(student, support.DATA_DIR, output)
else:
    print("先记录冻结方案，再执行最终测试。")

## 实验反馈
说明数据局限、准确率与召回率的差别；记录收获、难度和改进建议。正式报告按实验概况、C0–C7 实现与验证、综合实验结果、结果分析与总结四部分组织，导出 report.pdf；见 report-template.md。